# 04 — Train: Auto-ARIMA

Fits one univariate seasonal Auto-ARIMA model for the configured target station and evaluates it once on the test feature artifact.

**Inputs:** train-derived and test-derived feature artifacts  
**Outputs:** in-notebook prediction preview and test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and pins the notebook's constants: the bounds of the train-only order search, the daily seasonal period, the artifact paths and the Stage-3 column contract.

**What the imports provide**

- `auto_arima` — pmdarima's automatic order selection. It searches over `(p, d, q)(P, D, Q, m)` orders, fits each candidate, and keeps the one with the best information criterion. The fitted object wraps a statsmodels `SARIMAX` result, reachable as `.arima_res_`.
- `mean_absolute_error`, `root_mean_squared_error` — the two reported error metrics.
- `feature_column_names()`, `target_column_names()` — the Stage-3 column contract. Auto-ARIMA does not use these predictors; they only define the scoring cohort.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed` | Directory the Stage-3 feature Parquets are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | How many scored test rows the final preview table shows. Display only; it has no effect on any metric. |
| `FEATURE_COLUMNS` | 53 names | The frozen Stage-3 predictor contract, read from `feature_column_names()` instead of being hardcoded so the notebook fails loudly if Stage 3 ever changes it. It contains: the raw `water_level`, `imputed`, `precipitation` and `temperature_2m`; 8 water-level lags (1, 3, 6, 12, 24, 48, 72, 168 h); 5 water-level differences (1–24 h); 16 rolling water-level statistics (mean/std/min/max x 6/24/72/168 h); 4 rolling imputation counts; 4 rolling precipitation sums; 4 rolling temperature means plus the 24 h temperature min and max; and 6 calendar Fourier terms. |
| `TARGET_COLUMNS` | `target_t_plus_01` … `target_t_plus_24` | The 24 future water levels, one per lead hour. The model emits all of them from a single feature vector — a *direct* multi-horizon setup, with no recursive feeding of its own predictions. |
| `TRAIN_WATER_LEVEL_INTERPOLATION` | `linear`, `inside` | Settings for the temporary gap-filling applied to the *training input only*. `method="linear"` draws a straight line between the observations bracketing a gap. `limit_area="inside"` restricts that to gaps with an observation on **both** sides, so nothing is ever extrapolated past the first or last observation. The third key, `scope`, is documentation carried into the displayed table — it is not passed to pandas. |
| `AUTO_ARIMA_SEARCH` | see below | Every argument handed to `auto_arima`, kept in one dict so the exact search that was run can be displayed alongside its result. |

**The `AUTO_ARIMA_SEARCH` entries in detail**

| Key | Value | What it does |
| --- | --- | --- |
| `seasonal` | `True` | Allow a seasonal `(P, D, Q, m)` block in addition to the non-seasonal `(p, d, q)` one. |
| `m` | `24` | The seasonal period, in observations. The data is hourly, so 24 means one day. This is the single most expensive setting in the dict: seasonal terms at lag 24 widen the state space substantially. |
| `information_criterion` | `"aic"` | The score minimised when comparing candidate orders. AIC is `-2 x log-likelihood + 2k`, so it rewards fit and charges 2 per estimated parameter — a milder complexity penalty than BIC. Crucially it is computed **in-sample**, on the training series only, which is exactly why no held-out data is needed to choose an order. |
| `stepwise` | `True` | Use the Hyndman–Khandakar stepwise search: start from a handful of seed orders and walk to neighbouring ones as long as the criterion improves, instead of enumerating the whole grid. Orders of magnitude cheaper, with no guarantee of finding the grid optimum. |
| `max_p` / `max_q` | `5` | Upper bounds on the non-seasonal AR and MA orders. |
| `max_P` / `max_Q` | `2` | Upper bounds on the seasonal AR and MA orders at lag 24. Kept low deliberately — each one is expensive. |
| `max_d` | `2` | Upper bound on non-seasonal differencing. The actual `d` is not chosen by AIC but by repeated unit-root testing (KPSS by default). |
| `max_D` | `1` | Upper bound on seasonal differencing, likewise chosen by a seasonal-strength test (OCSB by default) rather than by AIC. |
| `max_order` | `None` | Disables the usual cap on `p + q + P + Q`. Under `stepwise=True` this is effectively a no-op, since the stepwise walk never enumerates a full grid anyway. |
| `out_of_sample_size` | `0` | Hold back no tail of the training series for validation. Selection is purely in-sample AIC — deliberate, because any held-out window would have to come out of the training period, and the test artifact must stay sealed. |
| `error_action` | `"raise"` | A candidate order that fails to fit aborts the notebook instead of being quietly skipped. Without this, a systematic fitting problem could disguise itself as "no better order was found". |
| `suppress_warnings` | `True` | Silences the convergence warnings statsmodels emits for the many rejected candidates. Only the noise is suppressed; genuine failures still raise, per the previous row. |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
PREDICTION_PREVIEW_ROWS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())
TRAIN_WATER_LEVEL_INTERPOLATION = {
    "method": "linear",
    "limit_area": "inside",
    "scope": "training input passed to auto_arima only",
}
AUTO_ARIMA_SEARCH = {
    "seasonal": True,
    "m": 24,
    "information_criterion": "aic",
    "stepwise": True,
    "max_p": 5,
    "max_q": 5,
    "max_P": 2,
    "max_Q": 2,
    "max_d": 2,
    "max_D": 1,
    "max_order": None,
    "out_of_sample_size": 0,
    "error_action": "raise",
    "suppress_warnings": True,
}

## Shared evaluation cohort

Auto-ARIMA is univariate: the only input it ever sees is the water-level series itself. It is nevertheless scored on exactly the same cohort as Ridge, the tree models and the persistence baseline — issue times that Stage 3 marked `target_valid` (all 24 future hours genuinely observed) **and** whose full 53-column predictor vector is present.

The predictors the model does not consume are therefore never handed to it; they exist here purely to define a comparable cohort. This costs some rows the model could technically have scored, and buys the ability to put this notebook's MAE next to the tree models' MAE without an asterisk.

## Helper functions

Four helpers used by the cells below. The keyword-only `station_id` / `artifact_name` arguments exist so that any raised error names the split it came from.

**`eligible_rows(frame, *, station_id, artifact_name) -> pd.Series`** — returns the boolean cohort mask described above, and raises if the artifact is missing a contract column or carries a null target inside a `target_valid` row.

**`water_level_series(frame, *, station_id, artifact_name) -> np.ndarray`** — returns the split's water level as a plain float array, but only after proving the series is safe to hand to a state-space model: single station, timezone-aware UTC timestamps, no duplicates, a contiguous hourly grid, numeric values, no infinities. `NaN` is permitted and preserved — see the next helper for what happens to it.

**`interpolated_train_water_levels(values) -> (prepared, missing_rows)`**

- `values` — the raw training water-level array, gaps included.
- Returns the gap-filled array plus the number of rows that were missing beforehand, so the run can report how much of its training input was reconstructed.

Interpolation is linear and `limit_area="inside"`, so only gaps flanked by real observations on both sides are filled. If a gap sits at either end of the series it cannot be filled and the helper raises rather than handing the search a `NaN`. This touches only the temporary array passed to `auto_arima`; the feature artifact on disk is never modified.

**`metric_tables(actual, predictions, *, station_id)`** and **`prediction_preview(frame, predictions)`** — identical to the other stage-4 notebooks. `metric_tables` returns one aggregate MAE/RMSE over all `n x 24` values plus the same pair per lead hour, which is what shows how fast accuracy decays from `t+1` to `t+24`. RMSE is always at least MAE and is dominated by the worst misses, so a wide gap between them means a few bad forecasts rather than uniformly poor ones.

In [ ]:
def eligible_rows(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> pd.Series:
    """Return model-ready rows and reject incomplete feature artifacts."""
    required_columns = {"timestamp", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    eligible = frame["target_valid"].eq(True) & frame[FEATURE_COLUMNS].notna().all(
        axis=1
    )
    if frame.loc[eligible, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return eligible

In [ ]:
def water_level_series(
    frame: pd.DataFrame, *, station_id: str, artifact_name: str
) -> np.ndarray:
    """Return one UTC-hourly station water-level series with no infinities."""
    required_columns = {"timestamp", "station_id", "water_level"}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")

    timestamp_dtype = frame["timestamp"].dtype
    if (
        not isinstance(timestamp_dtype, pd.DatetimeTZDtype)
        or str(timestamp_dtype.tz) != "UTC"
    ):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be timezone-aware UTC"
        )
    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.hasnans:
        raise ValueError(f"{station_id} {artifact_name} timestamps must be complete")
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if timestamps.has_duplicates or not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique, ascending, and hourly"
        )

    station_ids = frame["station_id"].drop_duplicates().tolist()
    if station_ids != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}; "
            f"got {station_ids!r}"
        )

    try:
        values = pd.to_numeric(frame["water_level"], errors="raise").to_numpy(
            dtype=float
        )
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must be numeric"
        ) from error
    if np.isinf(values).any():
        raise ValueError(
            f"{station_id} {artifact_name} water_level values must not contain infinities"
        )
    return values

In [ ]:
def interpolated_train_water_levels(values: np.ndarray) -> tuple[np.ndarray, int]:
    """Linearly fill internal training gaps for Auto-ARIMA only."""
    missing_rows = int(np.isnan(values).sum())
    interpolated = pd.Series(values).interpolate(
        method=TRAIN_WATER_LEVEL_INTERPOLATION["method"],
        limit_area=TRAIN_WATER_LEVEL_INTERPOLATION["limit_area"],
    )
    prepared = interpolated.to_numpy(dtype=float)
    if not np.isfinite(prepared).all():
        raise ValueError(
            "Auto-ARIMA training input has unfillable water_level gaps at a series boundary"
        )
    return prepared, missing_rows

In [ ]:
def metric_tables(
    actual: pd.DataFrame, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
                "rmse": root_mean_squared_error(
                    actual.to_numpy().ravel(), predictions.ravel()
                ),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[target], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[target], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(frame: pd.DataFrame, predictions: np.ndarray) -> pd.DataFrame:
    """Return issue timestamps, actual targets, and direct multi-step predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load and validate feature artifacts

Resolves `data/processed/<station>_train_features.parquet` and `<station>_test_features.parquet` for the station in `src.config.TARGET_STATION_ID`, raising `FileNotFoundError` if either is absent.

Then `water_level_series()` validates each split **independently**, because a state-space model consumes the raw series rather than a bag of rows, and a silently misaligned timeline would produce plausible-looking nonsense. The checks are that the artifact contains only the target station, that its timestamps are timezone-aware UTC, unique, ascending and form a contiguous hourly grid with no gaps, and that `water_level` is numeric and free of infinities.

`NaN` is explicitly allowed at this point. Values that Stage 2 filled in are flagged `imputed=True` but are real numbers; gaps too long to fill were deliberately preserved as missing, and they stay missing here. What happens to them differs by stage: in the training input they are interpolated (see below), while in the test loop they are appended to the filter as genuinely missing.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
for artifact_path in (train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(
            f"Missing feature artifact for {station_id}: {artifact_path}"
        )

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)
train_water_levels = water_level_series(
    train_features, station_id=station_id, artifact_name="train"
)
test_water_levels = water_level_series(
    test_features, station_id=station_id, artifact_name="test"
)

## Apply the eligibility cohort

Builds both masks, stops early if either split has no usable row, and keeps `test_rows` for scoring.

It also prepares `train_autoarima_values`: the training series with its internal gaps linearly interpolated. This copy exists only in memory and only as the search's input — the artifact on disk keeps its gaps. `interpolated_train_rows` records how many values were reconstructed, and is displayed in the next cell so the size of that intervention is visible rather than implied.

In [ ]:
train_mask = eligible_rows(train_features, station_id=station_id, artifact_name="train")
test_mask = eligible_rows(test_features, station_id=station_id, artifact_name="test")
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

test_rows = test_features.loc[test_mask]
train_autoarima_values, interpolated_train_rows = interpolated_train_water_levels(
    train_water_levels
)

## Fit the train-only Auto-ARIMA search

Unlike the ARIMAX and SARIMAX notebooks, which use a fixed hand-picked order, this one lets the data choose — but only within the bounds set in Setup and only from the training split. `auto_arima(train_autoarima_values, **AUTO_ARIMA_SEARCH)` fits each candidate order by maximum likelihood and keeps the one with the lowest AIC.

What is *not* used for selection: no validation split, no cross-validation, no held-out observations, no test rows, and none of the engineered predictors. The chosen order is then frozen for the rest of the notebook.

The three displayed tables are the audit trail — the selected order, seasonal order and AIC; the interpolation settings and how many rows they touched; and the full search configuration, so a future reader can tell which result came from which search without re-running anything.

In [ ]:
autoarima = auto_arima(train_autoarima_values, **AUTO_ARIMA_SEARCH)
search_result = pd.DataFrame(
    [
        {
            "station_id": station_id,
            "train_rows": len(train_water_levels),
            "interpolated_train_rows": interpolated_train_rows,
            "train_interpolation": "linear internal gaps only",
            "order": autoarima.order,
            "seasonal_order": autoarima.seasonal_order,
            "aic": autoarima.aic(),
        }
    ]
)
print(f"Auto-ARIMA train-only AIC search for {station_id}")
display(search_result)
display(pd.DataFrame([TRAIN_WATER_LEVEL_INTERPOLATION]))
display(pd.DataFrame([AUTO_ARIMA_SEARCH]))

## Issue rolling test forecasts

The parameters are fixed, but the *state* is not. Walking forward one hour at a time, each observed test water level is appended to the fitted filter and a fresh 24-step forecast is issued from there.

This is what makes the evaluation realistic: at 14:00 a forecaster genuinely knows the 14:00 reading, and refusing to use it would understate the model. What they do not know are the readings after 14:00 — and nothing in this loop uses them.

**The arguments that make this leakage-free**

- `state.append([water_level], refit=False)` — extends the fitted state with the newly observed hour. `refit=False` is the load-bearing argument: the coefficients stay exactly as estimated on the training split, and only the Kalman filter's state advances. With `refit=True` the model would re-estimate its parameters on data that includes test observations, and the reported score would be meaningless.
- A missing observation is appended as `NaN`. The Kalman filter treats it as an unobserved step and propagates its own prediction instead — no value is ever borrowed from a later hour to fill it.
- `get_forecast(steps=FORECAST_HORIZON_HOURS)` — `steps=24` produces exactly one value per lead hour from the current state, matching `TARGET_COLUMNS`. `.predicted_mean` takes the point forecast, i.e. the conditional mean, which is the quantity MAE and RMSE compare against.
- The shape check after each call turns a silent statsmodels shape change into an immediate failure.
- `np.vstack(all_test_predictions)[test_mask.to_numpy()]` — forecasts are issued at *every* test timestamp, because the filter has to walk through all of them in order to stay aligned, but only the cohort rows are kept for scoring.

In [ ]:
state = autoarima.arima_res_
all_test_predictions: list[np.ndarray] = []
for water_level in test_water_levels:
    state = state.append([water_level], refit=False)
    forecast = np.asarray(
        state.get_forecast(steps=FORECAST_HORIZON_HOURS).predicted_mean, dtype=float
    )
    if forecast.shape != (FORECAST_HORIZON_HOURS,):
        raise RuntimeError(
            f"Expected {FORECAST_HORIZON_HOURS} forecast values; got {forecast.shape}"
        )
    all_test_predictions.append(forecast)

test_predictions = np.vstack(all_test_predictions)[test_mask.to_numpy()]

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort: aggregate MAE/RMSE, the same two metrics per lead time, and a short preview so the predictions can be eyeballed against their actual targets. There is no second pass and no refitting — what is printed here is the notebook's one and only result.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS], test_predictions, station_id=station_id
)
print(f"Auto-ARIMA test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))